In [ ]:
# H3 Video Studio -- TEST MODE
# The jobs dataset carries test_mode.txt: render cheap placeholder clips with
# ffmpeg for every manifest entry, push them to the insurance dataset and stop.
# No GPU, no model downloads -- used to verify the studio loop end-to-end.
import json, os, shutil, subprocess, sys, time
from pathlib import Path

IN = Path("/kaggle/input")
manifest_path, base = None, None
# freshly created datasets take minutes to propagate AND may mount under a
# nested path (e.g. /kaggle/input/datasets/<slug>/...) -- scan recursively
for attempt in range(12):
    if IN.is_dir():
        for p in sorted(IN.glob("**/shots.json")):
            manifest_path, base = p, p.parent
            break
    if manifest_path:
        break
    tree = sorted(str(p) for p in IN.rglob("*"))[:20] if IN.is_dir() else ["<no /kaggle/input>"]
    print(f"[studio] waiting for jobs dataset ({attempt + 1}/12); input tree: {tree}", flush=True)
    time.sleep(55)
assert manifest_path, "shots.json never appeared under /kaggle/input"
shots = json.loads(manifest_path.read_text(encoding="utf-8"))
print(f"TEST MODE: {len(shots)} job(s) from {manifest_path}")

OUT = Path("/kaggle/working/out")
OUT.mkdir(parents=True, exist_ok=True)
t0 = time.time()
for s in shots:
    num, sec = int(s["num"]), int(s["seconds"])
    dest = OUT / f"shot_{num:02d}.mp4"
    cmd = ["ffmpeg", "-y", "-v", "error",
           "-f", "lavfi", "-i", f"color=c=0x1f2937:s=864x480:r=24:d={sec}",
           "-f", "lavfi", "-i", f"sine=frequency={330 + num * 40}:duration={sec}",
           "-vf", f"drawtext=text='TEST shot {num}':fontcolor=white@0.85:x=(w-tw)/2:y=(h-th)/2:fontsize=28",
           "-c:v", "libx264", "-preset", "veryfast", "-crf", "28",
           "-c:a", "aac", "-b:a", "96k", "-shortest", "-movflags", "+faststart",
           str(dest)]
    r = subprocess.run(cmd, capture_output=True, text=True)
    assert r.returncode == 0, f"ffmpeg failed for shot {num}: {r.stderr[-800:]}"
    print(f"  shot_{num:02d}.mp4  {sec}s  {dest.stat().st_size/1e6:.2f} MB")

# insurance push (same create-first pattern as the real kernel)
os.environ["KAGGLE_API_TOKEN"] = "KGAT_abba3bba83ed8f6a11855f7a819228ea"
(OUT / "dataset-metadata.json").write_text(json.dumps(
    {"title": "rmc-studio-outputs", "id": "freedomform/rmc-studio-outputs",
     "licenses": [{"name": "CC0-1.0"}]}, indent=1), encoding="utf-8")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle"], check=False)
mk = subprocess.run([sys.executable, "-m", "kaggle", "datasets", "create",
                     "-p", str(OUT), "--dir-mode", "skip"],
                    capture_output=True, text=True)
print("insurance create rc=", mk.returncode, (mk.stdout or mk.stderr)[-300:])
mv = subprocess.run([sys.executable, "-m", "kaggle", "datasets", "version",
                     "-p", str(OUT), "--dir-mode", "skip", "-m", f"studio test batch"],
                    capture_output=True, text=True)
print("insurance version rc=", mv.returncode, (mv.stdout or mv.stderr)[-300:])
print(f"TEST MODE DONE in {time.time()-t0:.0f}s -- {len(shots)} clip(s) rendered")
print("The remaining H3 stages are intentionally skipped in test mode.")
# NOTE: no os._exit here -- killing the kernel mid-run orphans the insurance
# upload and marks the session ERROR. The notebook simply ends after this cell.
